# LLM4Rec: SASRec to LLaMA Adapter Pipeline
Full pipeline for ECS172 project — runs on a Colab T4 GPU.

**Step order:**
1. Mount Drive & set path
2. Install dependencies
3. Train SASRec (Phase 1) → `checkpoints/item_embeddings.pt`
4. Train Adapter (Phase 2) → `checkpoints/adapter.pt`
5. Cache projected embeddings (one-shot, fast)
6. Run all 4 evaluations (text-only baseline + 3 injection modes)

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# IMPORTANT: Change this path to where your ECS172 folder is located in your Google Drive
PROJECT_DIR = '/content/drive/MyDrive/ECS172'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
!ls

## Cell 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
# Unsloth gives ~2x faster LLaMA loading/inference on Colab T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('Done.')

## Cell 3: (Optional) HuggingFace Login
Only needed if you want to use the gated `meta-llama/Llama-3.2-1B-Instruct` model.
Skip this cell if you use the default `unsloth/Llama-3.2-1B-Instruct` (no token required).

In [ ]:
# from huggingface_hub import login
# login("YOUR_HF_TOKEN_HERE")

## Cell 4: Train SASRec (Phase 1)
Creates `checkpoints/item_embeddings.pt` and `checkpoints/id_maps.json`.
Only needs to run once. ~5-10 min on T4.

In [ ]:
!python scripts/train_sasrec.py --epochs 15

## Cell 5: Train Adapter (Phase 2)
Trains the MLP projector using LLaMA-grounded teacher-forced CE loss on movie titles.
Creates `checkpoints/adapter.pt`.

**To use your fine-tuned LLaMA:** Change `--llm-model` to `./llama31-1b-movielens-full-final`

In [ ]:
!python scripts/train_adapter.py \
    --epochs 10 \
    --batch-size 8 \
    --llm-model unsloth/Llama-3.2-1B-Instruct

## Cell 6: Cache Projected Embeddings
Runs the adapter over ALL movies once and saves the results to disk.
This is fast (~seconds) and means the evaluation loop never needs to call the adapter again.
Creates `checkpoints/projected_embeddings.pt` of shape `(n_items, llm_dim)`.

In [ ]:
!python scripts/cache_projected_embeddings.py

## Cell 7: Build A-J Prompts, Sanity Check, Run Mode A vs Mode C

In [ ]:
# Cell 7: Build prompts (A-J) + pre-flight sanity, then run Mode A vs Mode C
LLM_MODEL = 'unsloth/Llama-3.2-1B-Instruct'  # or './llama31-1b-movielens-full-final'

# 1) Materialise the letter-based prompt artifact (torch-free, verifies off-by-one)
!python scripts/build_AC_prompts.py

# 2) Pre-flight: cache round-trip + single-token letters (catches silent failures)
!python scripts/sanity_check.py --model $LLM_MODEL

# 3) Evaluate both conditions against one loaded model:
#      Mode A  = text baseline (candidate titles as text)
#      Mode C  = candidate soft tokens (cached adapter output) + letter labels
!python scripts/eval_ranking.py --model $LLM_MODEL --modes A C --n-candidates 10


## Cell 8: Compare Results (A vs C + random baseline)

In [ ]:
# Cell 8: Compare Mode A vs Mode C (with random baseline)
import json
from pathlib import Path

conditions = [
    ('Mode A (text baseline)',         'results_mode_A.json'),
    ('Mode C (candidate soft tokens)', 'results_mode_C.json'),
]

ks, rand = [], {}
for _, f in conditions:
    pth = Path(f)
    if pth.exists():
        data = json.loads(pth.read_text())
        ks = sorted(int(k.split('@')[1]) for k in data['metrics'] if k.startswith('HR@'))
        rand = data.get('random_baseline', {})
        break

def fmt_row(label, m):
    cells = ''.join(f"{m.get(f'HR@{k}', 0):>8.3f}" for k in ks)
    cells += ''.join(f"{m.get(f'NDCG@{k}', 0):>9.4f}" for k in ks)
    return f"{label:<32}{cells}"

header = f"{'Condition':<32}" + ''.join(f"{'HR@'+str(k):>8}" for k in ks) + ''.join(f"{'NDCG@'+str(k):>9}" for k in ks)
print(header)
print('-' * len(header))
if rand:
    print(fmt_row('Random', rand))
for label, fname in conditions:
    pth = Path(fname)
    if not pth.exists():
        print(f"{label:<32}  (not run yet)")
        continue
    print(fmt_row(label, json.loads(pth.read_text())['metrics']))
